# Adv_graphRAG: Hybrid, Citation-Grounded GraphRAG over OpenFDA labels
Pipeline: **chunk + stamp -> embed (nomic) -> BM25 + FAISS + RRF -> graph traversal (drug->section only) -> token-budget context with node IDs -> Llama-3 answer -> support check + abstention -> QA evaluation**.

Run cells top to bottom. Every step prints what it is doing. HyDE is included but **OFF** (`USE_HYDE=False`).
Use a GPU runtime (T4) or embedding/generation will be very slow. Outputs are saved to `.../graph_rag/Adv_graphRAG/`.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
from google.colab import drive
import os, re, json, time, pickle, hashlib, html

drive.mount('/content/drive')

# --- guard: refuse to continue unless Drive is REALLY mounted ---
def drive_is_mounted():
    try:
        return any(l.split()[1] == "/content/drive" for l in open("/proc/mounts"))
    except Exception:
        return os.path.ismount("/content/drive")

if not drive_is_mounted():
    raise RuntimeError("Google Drive is NOT mounted. Re-run this cell and approve the pop-up. "
                       "Nothing was created or saved.")
print("Drive mounted: OK")

BASE_DIR = "/content/drive/MyDrive/Research_kgTxagent/graph_rag"
if not os.path.isdir(BASE_DIR):
    raise RuntimeError(f"{BASE_DIR} not found on Drive. Check the folder name; refusing to create it.")
OUT_DIR = os.path.join(BASE_DIR, "Adv_graphRAG")
os.makedirs(OUT_DIR, exist_ok=True)          # only reached when Drive is confirmed mounted
print("Outputs go to:", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted: OK
Outputs go to: /content/drive/MyDrive/Research_kgTxagent/graph_rag/Adv_graphRAG


In [ ]:
# ===== STEP 1: system + python packages =====
print("STEP 1: installing packages ...")
!apt-get update -qq && apt-get install -y -qq zstd
!pip install -q ollama faiss-cpu bm25s networkx pandas numpy tqdm requests
print("STEP 1 DONE")

STEP 1: installing packages ...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
STEP 1 DONE


In [ ]:
# ===== STEP 2: install Ollama, start server, pull models =====
import subprocess, time, requests, os
print("STEP 2a: installing Ollama ...")
r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
print("install return code:", r.returncode)
print(r.stdout[-400:], r.stderr[-400:])

print("STEP 2b: starting server ...")
log = open("ollama.log", "w")
subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log)
ready = False
for i in range(60):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            ready = True; break
    except Exception:
        pass
    time.sleep(1)
print("server ready:", ready, "(waited", i + 1, "s)")
if not ready:
    print(open("ollama.log").read()[-1500:])
    raise RuntimeError("Ollama server did not start - see log above")

print("STEP 2c: pulling models (a few minutes the first time) ...")
for m in ["llama3:8b", "nomic-embed-text"]:
    p = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print(m, "-> return code", p.returncode, p.stdout[-120:].strip(), p.stderr[-200:].strip())
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

STEP 2a: installing Ollama ...
install return code: 0
 0%
#######################################################################   99.6%
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

STEP 2b: starting server ...
server ready: True (waited 1 s)
STEP 2c: pulling models (a few minutes the first time) ...
llama3:8b -> return code 0  
pulling 3f8eb4da87fa: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 
nomic-embed-text -> return code 0  
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 
NAME                       ID              SIZE

In [ ]:
import os, re, json, time, pickle, hashlib, html
from collections import defaultdict, Counter
import numpy as np, pandas as pd, requests, faiss, bm25s
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

BASE_DIR = "/content/drive/MyDrive/Research_kgTxagent/graph_rag"
OUT_DIR  = os.path.join(BASE_DIR, "Adv_graphRAG")
os.makedirs(OUT_DIR, exist_ok=True)
print("Outputs go to:", OUT_DIR)

def first_existing(paths):
    for p in paths:
        if os.path.exists(p): return p
    return None

GRAPH_PATH = first_existing([
    "/content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl",
    os.path.join(BASE_DIR, "OpenFDA_graph.pkl"),
    os.path.join(BASE_DIR, "graph_with_embeddings.pkl"),
])
QA_PATH = first_existing([
    os.path.join(OUT_DIR, "QA.json"), os.path.join(BASE_DIR, "QA.json"),
    "/content/QA.json", "/content/drive/MyDrive/QA.json",
])
print("GRAPH_PATH:", GRAPH_PATH)
print("QA_PATH   :", QA_PATH, "(if None, upload QA.json to", OUT_DIR, "before the QA step)")

# ---- models ----
LLM_MODEL, EMBED_MODEL = "llama3:8b", "nomic-embed-text"
EMBED_URL, GEN_URL = "http://localhost:11434/api/embed", "http://localhost:11434/api/generate"

# ---- chunking ----
EXCLUDE_SECTIONS = {"inactive_ingredient"}   # noisy ingredient lists; remove from set to include
MAX_CHARS, OVERLAP_UNITS = 700, 1            # chunk size; units shared between neighbouring chunks

# ---- embedding ----
EMB_BATCH, EMB_WORKERS, SHARD_SIZE = 64, 4, 4000

# ---- retrieval ----
RRF_K, MAX_DRUGS, SYMPTOM_FETCH = 60, 5, 60
TOTAL_CONTEXT_TOKENS, CHARS_PER_TOKEN = 3000, 4
ABSTAIN_COS = 0.40        # symptom mode: best chunk cosine below this -> abstain (tune after first run)
USE_HYDE = False          # query expansion, optional last experiment

# ---- QA run ----
SKIP_PREFIXES = ()   # T7 was dropped
print("Config loaded.")

Outputs go to: /content/drive/MyDrive/Research_kgTxagent/graph_rag/Adv_graphRAG
GRAPH_PATH: /content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl
QA_PATH   : None (if None, upload QA.json to /content/drive/MyDrive/Research_kgTxagent/graph_rag/Adv_graphRAG before the QA step)
Config loaded.


In [ ]:
# ===== STEP 4: Ollama helpers =====
DIM = None
FAILED_EMBEDS = []

def _embed_call(texts):
    r = requests.post(EMBED_URL, json={"model": EMBED_MODEL, "input": texts, "truncate": True}, timeout=300)
    r.raise_for_status()
    return r.json()["embeddings"]

def embed_texts(texts, prefix):
    # batch call; if it fails, retry one by one with shorter text (this fixes the old HTTP 500 failures)
    texts = [prefix + t for t in texts]
    try:
        return np.array(_embed_call(texts), dtype="float32")
    except Exception as e:
        out = []
        for t in texts:
            vec = None
            for limit in (len(t), 1500, 800, 400):
                try:
                    vec = _embed_call([t[:limit]])[0]; break
                except Exception as e2:
                    last = e2
            if vec is None:
                FAILED_EMBEDS.append(t[:80]); print("  !! embed failed for:", t[:60], "|", last)
                vec = [0.0] * DIM
            out.append(vec)
        return np.array(out, dtype="float32")

def llm(prompt, temperature=0.1, num_predict=500):
    r = requests.post(GEN_URL, json={"model": LLM_MODEL, "prompt": prompt, "stream": False,
        "options": {"temperature": temperature, "num_ctx": 8192, "num_predict": num_predict}}, timeout=600)
    r.raise_for_status()
    return r.json()["response"].strip()

print("Testing embedding ...")
v = embed_texts(["hello"], "search_query: ")
DIM = v.shape[1]
print("embedding dim:", DIM)
print("Testing LLM ...")
print("LLM says:", llm("Reply with the single word OK.", num_predict=5))

Testing embedding ...
embedding dim: 768
Testing LLM ...
LLM says: OK


In [ ]:
!nvidia-smi
!ollama ps

Tue Nov 10 03:09:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             34W /   70W |    6007MiB /  15360MiB |     13%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ===== STEP 5: load graph and build lookup tables =====
print("Loading graph:", GRAPH_PATH)
t0 = time.time()
with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)
print(f"Loaded in {time.time()-t0:.0f}s | nodes={G.number_of_nodes():,} edges={G.number_of_edges():,} directed={G.is_directed()}")

type_counts = Counter(d.get("type") for _, d in G.nodes(data=True))
print("Node types:", dict(type_counts))
SECTION_TYPES = sorted(t for t in type_counts if t not in ("drug_generic", "drug_brand"))
print("Section types:", SECTION_TYPES)

generic_ids = [n for n, d in G.nodes(data=True) if d.get("type") == "drug_generic"]
brand_ids   = [n for n, d in G.nodes(data=True) if d.get("type") == "drug_brand"]
GENERIC_NAME = {n: G.nodes[n].get("name", "") for n in generic_ids}
BRAND_NAME   = {n: G.nodes[n].get("name", "") for n in brand_ids}

# only drug->section edges are used (section<->section edges are a construction artifact)
BRAND_TO_GENERIC, DRUG_SECTIONS = defaultdict(list), defaultdict(list)
for g in tqdm(generic_ids, desc="indexing drug->section edges"):
    for nb in G.neighbors(g):
        t = G.nodes[nb].get("type")
        if t == "drug_brand": BRAND_TO_GENERIC[nb].append(g)
        elif t in SECTION_TYPES: DRUG_SECTIONS[g].append((nb, t))
print("generics:", len(generic_ids), "| brands:", len(brand_ids), "| brands with a generic parent:", len(BRAND_TO_GENERIC))
g0 = generic_ids[0]
print("sample generic:", GENERIC_NAME[g0], "->", [t for _, t in DRUG_SECTIONS[g0]])

Loading graph: /content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl
Loaded in 8s | nodes=110,843 edges=355,909 directed=False
Node types: {'drug_generic': 10840, 'drug_brand': 31797, 'dosage_and_administration': 9443, 'dosage_and_administration_table': 1454, 'indications_and_usage': 9442, 'active_ingredient': 7008, 'inactive_ingredient': 7107, 'keep_out_of_reach_of_children': 6985, 'purpose': 6871, 'warnings': 7590, 'do_not_use': 2082, 'stop_use': 2814, 'ask_doctor': 1585, 'pregnancy_or_breast_feeding': 2388, 'storage_and_handling': 2688, 'ask_doctor_or_pharmacist': 749}
Section types: ['active_ingredient', 'ask_doctor', 'ask_doctor_or_pharmacist', 'do_not_use', 'dosage_and_administration', 'dosage_and_administration_table', 'inactive_ingredient', 'indications_and_usage', 'keep_out_of_reach_of_children', 'pregnancy_or_breast_feeding', 'purpose', 'stop_use', 'storage_and_handling', 'warnings']


indexing drug->section edges:   0%|          | 0/10840 [00:00<?, ?it/s]

generics: 10840 | brands: 31797 | brands with a generic parent: 31797
sample generic: LIDOCAINE -> ['active_ingredient', 'do_not_use', 'dosage_and_administration', 'inactive_ingredient', 'indications_and_usage', 'keep_out_of_reach_of_children', 'pregnancy_or_breast_feeding', 'purpose', 'warnings']


In [ ]:
# ===== STEP 6: section-aware chunking + drug stamping =====
def section_text(sid):
    txt = G.nodes[sid].get("description") or ""
    txt = html.unescape(re.sub(r"<[^>]+>", " ", txt))
    return re.sub(r"\s+", " ", txt).strip()

def split_units(text):
    # split on ' ; ', bullets and sentence ends (the text is joined with ' ; ', so NLTK would fail)
    parts = re.split(r"\s;\s|\s•\s|(?<=[.!?])\s+(?=[A-Z])", text)
    units = []
    for p in parts:
        p = p.strip()
        while len(p) > MAX_CHARS:                       # hard-split very long units on a word boundary
            cut = p.rfind(" ", 0, MAX_CHARS); cut = cut if cut > 100 else MAX_CHARS
            units.append(p[:cut].strip()); p = p[cut:].strip()
        if p: units.append(p)
    return units

def window_units(units):
    chunks, cur, cur_len = [], [], 0
    for u in units:
        if cur and cur_len + len(u) + 3 > MAX_CHARS:
            chunks.append(" ; ".join(cur))
            keep = cur[-OVERLAP_UNITS:] if (OVERLAP_UNITS > 0 and len(cur) >= 3) else []
            if keep and sum(len(x) for x in keep) + len(u) + 3 > MAX_CHARS: keep = []
            cur, cur_len = list(keep), sum(len(x) + 3 for x in keep)
        cur.append(u); cur_len += len(u) + 3
    if cur: chunks.append(" ; ".join(cur))
    return chunks

CHUNKS, skipped_empty = [], 0
for g in tqdm(generic_ids, desc="chunking"):
    for sid, stype in DRUG_SECTIONS[g]:
        if stype in EXCLUDE_SECTIONS: continue
        txt = section_text(sid)
        if not txt: skipped_empty += 1; continue
        for k, piece in enumerate(window_units(split_units(txt))):
            CHUNKS.append({"cid": f"{sid}#{k}", "section_id": sid, "section": stype, "drug_id": g,
                           "drug": GENERIC_NAME[g], "text": piece,
                           "stamped": f"[DRUG: {GENERIC_NAME[g]} | SECTION: {stype}] {piece}"})
print(f"chunks={len(CHUNKS):,} | empty sections skipped={skipped_empty:,} | avg chars={np.mean([len(c['text']) for c in CHUNKS]):.0f}")
print("chunks per section:", dict(Counter(c["section"] for c in CHUNKS).most_common()))
for c in CHUNKS[:3]: print("SAMPLE:", c["cid"], "|", c["stamped"][:200])

SIG = hashlib.md5(f"{MAX_CHARS}-{OVERLAP_UNITS}-{sorted(EXCLUDE_SECTIONS)}-{len(CHUNKS)}".encode()).hexdigest()[:8]
with open(os.path.join(OUT_DIR, f"chunks_{SIG}.pkl"), "wb") as f: pickle.dump(CHUNKS, f)
print("signature:", SIG, "(embedding shards are tied to this, so changing chunk settings re-embeds)")

chunking:   0%|          | 0/10840 [00:00<?, ?it/s]

chunks=97,579 | empty sections skipped=0 | avg chars=347
chunks per section: {'dosage_and_administration': 30531, 'warnings': 15793, 'indications_and_usage': 13125, 'active_ingredient': 7328, 'purpose': 7070, 'keep_out_of_reach_of_children': 7006, 'dosage_and_administration_table': 4122, 'stop_use': 2844, 'storage_and_handling': 2812, 'pregnancy_or_breast_feeding': 2393, 'do_not_use': 2197, 'ask_doctor': 1606, 'ask_doctor_or_pharmacist': 752}
SAMPLE: SECTION:7ce1aab622816ca5#0 | [DRUG: LIDOCAINE | SECTION: active_ingredient] ACTIVE INGREDIENTS: Lidocaine 4.00% Menthol 1.00% Topical Anesthetic External Analgesic
SAMPLE: SECTION:844ec1ce62df85da#0 | [DRUG: LIDOCAINE | SECTION: do_not_use] Do not use in large quantities, particularly over raw surfaces or blistered areas.
SAMPLE: SECTION:4d48901dd8df4009#0 | [DRUG: LIDOCAINE | SECTION: dosage_and_administration] DIRECTIONS (Adults and Children Over 12 Years): Clean and dry affected area. ; Remove patch from backing and apply to affected ar

In [ ]:
# ===== STEP 7: embed chunks (sharded + resumable) =====
SHARD_DIR = os.path.join(OUT_DIR, f"shards_{SIG}")
os.makedirs(SHARD_DIR, exist_ok=True)
n_shards = (len(CHUNKS) + SHARD_SIZE - 1) // SHARD_SIZE
print(f"{len(CHUNKS):,} chunks -> {n_shards} shards of {SHARD_SIZE}. Finished shards are reused if you re-run.")

def embed_batch(batch):
    return embed_texts([c["stamped"] for c in batch], "search_document: ")

shard_arrays = []
for s in range(n_shards):
    path = os.path.join(SHARD_DIR, f"emb_{s:04d}.npy")
    if os.path.exists(path):
        shard_arrays.append(np.load(path)); print(f"shard {s+1}/{n_shards}: loaded from disk"); continue
    t0 = time.time()
    part = CHUNKS[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
    batches = [part[i:i + EMB_BATCH] for i in range(0, len(part), EMB_BATCH)]
    with ThreadPoolExecutor(EMB_WORKERS) as ex:
        res = list(tqdm(ex.map(embed_batch, batches), total=len(batches), desc=f"shard {s+1}/{n_shards}", leave=False))
    arr = np.vstack(res).astype("float32")
    np.save(path, arr); shard_arrays.append(arr)
    print(f"shard {s+1}/{n_shards}: {arr.shape} in {time.time()-t0:.0f}s | failed so far: {len(FAILED_EMBEDS)}")
    print("  GPU util %, mem MiB:", subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
    print("  ollama ps:", subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout.strip())

EMB = np.ascontiguousarray(np.vstack(shard_arrays), dtype="float32")
print("EMB shape:", EMB.shape, "| zero vectors:", int((np.abs(EMB).sum(1) == 0).sum()))
assert EMB.shape[0] == len(CHUNKS), "embedding count != chunk count"
faiss.normalize_L2(EMB)
INDEX = faiss.IndexFlatIP(EMB.shape[1]); INDEX.add(EMB)
faiss.write_index(INDEX, os.path.join(OUT_DIR, f"faiss_{SIG}.index"))
print("FAISS index:", INDEX.ntotal, "vectors saved.")

97,579 chunks -> 25 shards of 4000. Finished shards are reused if you re-run.
shard 1/25: loaded from disk
shard 2/25: loaded from disk
shard 3/25: loaded from disk
shard 4/25: loaded from disk
shard 5/25: loaded from disk
shard 6/25: loaded from disk
shard 7/25: loaded from disk
shard 8/25: loaded from disk
shard 9/25: loaded from disk
shard 10/25: loaded from disk
shard 11/25: loaded from disk
shard 12/25: loaded from disk
shard 13/25: loaded from disk


shard 14/25:   0%|          | 0/63 [00:00<?, ?it/s]

shard 14/25: (4000, 768) in 59s | failed so far: 0
  GPU util %, mem MiB: 44 %, 6017 MiB
  ollama ps: NAME                       ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
nomic-embed-text:latest    0a109f422b47    335 MB    100% GPU     2048       llamacpp    4 minutes from now    
llama3:8b                  365c0bd3c000    5.7 GB    100% GPU     8192       llamacpp    3 minutes from now


shard 15/25:   0%|          | 0/63 [00:00<?, ?it/s]

shard 15/25: (4000, 768) in 39s | failed so far: 0
  GPU util %, mem MiB: 18 %, 6017 MiB
  ollama ps: NAME                       ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
nomic-embed-text:latest    0a109f422b47    335 MB    100% GPU     2048       llamacpp    4 minutes from now    
llama3:8b                  365c0bd3c000    5.7 GB    100% GPU     8192       llamacpp    2 minutes from now


shard 16/25:   0%|          | 0/63 [00:00<?, ?it/s]

... (15 outputs left out)

shard 24/25:   0%|          | 0/63 [00:00<?, ?it/s]

shard 24/25: (4000, 768) in 41s | failed so far: 0
  GPU util %, mem MiB: 50 %, 465 MiB
  ollama ps: NAME                       ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
nomic-embed-text:latest    0a109f422b47    335 MB    100% GPU     2048       llamacpp    4 minutes from now


shard 25/25:   0%|          | 0/25 [00:00<?, ?it/s]

shard 25/25: (1579, 768) in 16s | failed so far: 0
  GPU util %, mem MiB: 49 %, 465 MiB
  ollama ps: NAME                       ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
nomic-embed-text:latest    0a109f422b47    335 MB    100% GPU     2048       llamacpp    4 minutes from now
EMB shape: (97579, 768) | zero vectors: 0
FAISS index: 97579 vectors saved.


In [ ]:
# ===== STEP 8: BM25 index + lookup tables =====
STOP = {"the","a","an","of","and","to","is","are","can","i","if","for","in","on","or","with","what","do","how","my","me","it"}
def tokenize(s):
    s = re.sub(r"(\d)([a-z])", r"\1 \2", s.lower())
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in STOP]

print("tokenizing + building BM25 (stamped text, so the drug name is searchable) ...")
t0 = time.time()
BM25 = bm25s.BM25()
BM25.index([tokenize(c["stamped"]) for c in CHUNKS], show_progress=False)
print(f"BM25 ready in {time.time()-t0:.0f}s")

IDX_BY_SECTION, IDX_BY_DRUG = defaultdict(list), defaultdict(list)
for i, c in enumerate(CHUNKS):
    IDX_BY_SECTION[c["section"]].append(i); IDX_BY_DRUG[c["drug_id"]].append(i)
IDX_BY_SECTION = {k: np.array(v) for k, v in IDX_BY_SECTION.items()}
IDX_BY_DRUG = {k: np.array(v) for k, v in IDX_BY_DRUG.items()}

# self-test
sc = BM25.get_scores(tokenize("lidocaine pregnancy"))
top = np.argsort(-sc)[:3]
print("BM25 self-test 'lidocaine pregnancy':")
for i in top: print("  ", round(float(sc[i]), 2), CHUNKS[i]["cid"], CHUNKS[i]["stamped"][:110])

tokenizing + building BM25 (stamped text, so the drug name is searchable) ...


DEBUG:bm25s:Building index from tokens


BM25 ready in 13s
BM25 self-test 'lidocaine pregnancy':
   4.17 SECTION:c5fc2106fc5b3fd4#0 [DRUG: 4% LIDOCAINE | SECTION: pregnancy_or_breast_feeding] ask a health professional before use.
   4.12 SECTION:d98174bd60219e5b#0 [DRUG: LIDOCAINE 5% | SECTION: pregnancy_or_breast_feeding] Ask a health care professional before use.
   4.07 SECTION:4a542c0e34ccd04b#0 [DRUG: LIDOCAINE SPRAY | SECTION: pregnancy_or_breast_feeding] If pregnant or breastfeeding , ask a health pro


In [ ]:
# ===== STEP 9: hybrid retrieval (dense + BM25 + RRF) =====
_QCACHE = {}
def qvec(q):
    if q not in _QCACHE:
        v = embed_texts([q], "search_query: ")[0]; _QCACHE[q] = v / (np.linalg.norm(v) + 1e-9)
    return _QCACHE[q]

def dense_topk(qv, allowed, k):
    if allowed is None:                                   # unfiltered: FAISS
        _, I = INDEX.search(qv.reshape(1, -1).astype("float32"), k)
        return [int(i) for i in I[0] if i >= 0]
    sims = EMB[allowed] @ qv                              # filtered: exact scoring on the subset
    return [int(allowed[o]) for o in np.argsort(-sims)[:k]]

def bm25_topk(q, allowed, k):
    toks = tokenize(q)
    if not toks: return []
    scores = BM25.get_scores(toks)
    if allowed is None:
        cand = np.argpartition(-scores, min(k, len(scores) - 1))[:k]; cand = cand[np.argsort(-scores[cand])]
    else:
        cand = allowed[np.argsort(-scores[allowed])[:k]]
    return [int(c) for c in cand if scores[c] > 0]

def rrf(lists, weights=None):
    out = defaultdict(float)
    for li, lst in enumerate(lists):
        w = 1.0 if weights is None else weights[li]
        for rank, idx in enumerate(lst, start=1): out[idx] += w / (RRF_K + rank)
    return out

def hybrid_rank(queries, allowed=None, k=50):
    # returns [(chunk_idx, rrf_score, best_cosine)] sorted by rrf
    lists, qvs = [], []
    for q in queries:
        qv = qvec(q); qvs.append(qv)
        lists.append(dense_topk(qv, allowed, k)); lists.append(bm25_topk(q, allowed, k))
    fused = sorted(rrf(lists).items(), key=lambda x: -x[1])[:k]
    return [(i, s, max(float(EMB[i] @ qv) for qv in qvs)) for i, s in fused]

def expand_queries(q):
    if not USE_HYDE: return [q]
    hyde = llm("Write 2 sentences an FDA drug label might contain that answer this question. "
               f"Do not name a drug unless it is in the question.\nQuestion: {q}", temperature=0.2, num_predict=120)
    print("  HyDE text:", hyde); return [q, hyde]

r = hybrid_rank(["what can I take for a headache"], k=5)
print("hybrid self-test 'what can I take for a headache':")
for i, s, c in r: print(f"  rrf={s:.4f} cos={c:.3f} {CHUNKS[i]['drug']} | {CHUNKS[i]['section']}")

hybrid self-test 'what can I take for a headache':
  rrf=0.0164 cos=0.820 BELLADONNA, BRYONIA (ALBA), GELSEMIUM SEMPERVIRENS, GLONOINUM, MELILOTUS OFFICINALIS, NUX VOMICA, SILICEA, SPIGELIA ANTHELMIA | indications_and_usage
  rrf=0.0164 cos=0.779 CALENDULA OFFICINALIS, DULCAMARA, EQUISETUM ARVENSE, HYPERICUM PERFORATUM | indications_and_usage
  rrf=0.0161 cos=0.817 BELLADONNA, BRYONIA (ALBA), GELSEMIUM SEMPERVIRENS, GLONOINUM, MELILOTUS OFFICINALIS, NUX VOMICA, SILICEA, SPIGELIA ANTHELMIA | purpose
  rrf=0.0161 cos=0.796 EXTRA STRENGHT HEADACHE RELIEF- ACETAMINOPHEN, ASPIRIN, CAFFEINE TABLET | warnings
  rrf=0.0159 cos=0.816 ARABICA COFFEE BEAN - STRYCHNOS NUX-VOMICA SEED - DELPHINIUM STAPHISAGRIA SEED | purpose


In [ ]:
# ===== STEP 10: entity detection (data-driven) + intent -> sections =====
def norm_tokens(s):
    s = re.sub(r"(\d)([a-z])", r"\1 \2", s.lower())
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in {"and", "the", "of", "with"}]

PRECISION_MIN = 0.15   # rule threshold; decide BEFORE looking at QA results and then freeze it

# --- 1. which drugs have each word in their NAME (generic names + brand names via parent) ---
print("10a: building name vocabulary ...")
name_vocab = defaultdict(set)
for g, nm in GENERIC_NAME.items():
    for t in set(norm_tokens(nm)): name_vocab[t].add(g)
for b, nm in BRAND_NAME.items():
    gens = BRAND_TO_GENERIC.get(b, [])
    for t in set(norm_tokens(nm)): name_vocab[t].update(gens)
print("  distinct name words:", len(name_vocab))

# --- 2. which drugs mention each word in their label TEXT ---
print("10b: scanning label text (about a minute) ...")
text_vocab = defaultdict(set)
for c in tqdm(CHUNKS, desc="scan chunks"):
    for t in set(norm_tokens(c["text"])):
        if t in name_vocab: text_vocab[t].add(c["drug_id"])

# --- 3. precision of each word ---
TOK_PREC = {}
for t, name_drugs in name_vocab.items():
    td = text_vocab.get(t, set())
    TOK_PREC[t] = (len(name_drugs & td) / len(td)) if td else 1.0
print("  words with precision < PRECISION_MIN:", sum(p < PRECISION_MIN for p in TOK_PREC.values()))

# --- 4. build entity table, dropping entities with no specific word ---
ENT, dropped = [], []
for g, nm in GENERIC_NAME.items():
    ts = frozenset(norm_tokens(nm))
    if not ts: continue
    spec = max(TOK_PREC[t] for t in ts)
    (ENT if spec >= PRECISION_MIN else dropped).append((ts, nm, "generic", g) if spec >= PRECISION_MIN else (nm, round(spec, 3)))
for b, nm in BRAND_NAME.items():
    if not BRAND_TO_GENERIC.get(b): continue
    ts = frozenset(norm_tokens(nm))
    if not ts: continue
    spec = max(TOK_PREC[t] for t in ts)
    (ENT if spec >= PRECISION_MIN else dropped).append((ts, nm, "brand", b) if spec >= PRECISION_MIN else (nm, round(spec, 3)))
print(f"entity table: kept={len(ENT):,} | dropped as too generic={len(dropped):,}")
print("  sample of dropped (name, precision):", sorted(dropped, key=lambda x: x[1])[:12])

def detect_entities(query):
    qt = set(norm_tokens(query)); hits = []
    for ts, nm, kind, eid in ENT:
        if ts <= qt or (len(ts) >= 4 and len(ts & qt) / len(ts) >= 0.8): hits.append((ts, nm, kind, eid))
    keep = [h for h in hits if not any(h[0] < o[0] for o in hits)]       # drop names contained in a longer matched name
    keep.sort(key=lambda h: -len(h[0]))
    out, seen = [], set()
    for ts, nm, kind, eid in keep:
        gens = [eid] if kind == "generic" else BRAND_TO_GENERIC.get(eid, [])
        for g in gens:
            if g not in seen: seen.add(g); out.append({"drug_id": g, "matched": nm, "kind": kind})
    return out[:MAX_DRUGS]

INTENT_RULES = [
 (r"pregnan|breast|nursing|lactat|unborn", ["pregnancy_or_breast_feeding", "warnings"]),
 (r"\bchild|\bkids?\b|infant|baby|toddler|teen|year.old|under \d+|pediatric", ["do_not_use", "dosage_and_administration", "warnings"]),
 (r"how (many|much|often|long)|times a day|\bdose|dosage|\bapply|maximum|per day|24 hours|every \d+", ["dosage_and_administration", "dosage_and_administration_table"]),
 (r"stop (using|use)|when should i stop|discontinue", ["stop_use", "warnings"]),
 (r"do not use|should not use|who should not|\bavoid|contraindic|exclusion", ["do_not_use", "warnings"]),
 (r"ask (a )?doctor|consult|before (use|using|taking)|medical condition|\bliver\b|\bkidney\b|\bheart\b|thyroid|diabet", ["ask_doctor", "ask_doctor_or_pharmacist", "warnings"]),
 (r"warning|serious|reaction|side effect|\brisk|alert|allerg", ["warnings", "stop_use"]),
 (r"\bstor(e|age)|temperature", ["storage_and_handling"]),
 (r"used for|treat|relie|indicat|help(s)? with|what can i (take|use)|remed|products|what is", ["indications_and_usage", "purpose"]),
]
DEFAULT_SECTIONS_DRUG = ["indications_and_usage", "warnings", "do_not_use", "ask_doctor", "stop_use",
                         "pregnancy_or_breast_feeding", "dosage_and_administration"]
SECTION_ORDER = ["indications_and_usage", "purpose", "active_ingredient", "dosage_and_administration",
                 "dosage_and_administration_table", "warnings", "do_not_use", "ask_doctor", "ask_doctor_or_pharmacist",
                 "stop_use", "pregnancy_or_breast_feeding", "keep_out_of_reach_of_children", "storage_and_handling"]

def detect_intent(query):
    q, secs = query.lower(), []
    for pat, s in INTENT_RULES:
        if re.search(pat, q):
            for x in s:
                if x not in secs: secs.append(x)
    return secs

for q in ["What should I avoid if I am pregnant and using lidocaine?", "Is Aspercreme safe for children?",
          "What can I take for headaches while pregnant?"]:
    print(q, "\n   entities:", [(e["matched"], e["kind"]) for e in detect_entities(q)], "| intent:", detect_intent(q))

10a: building name vocabulary ...
  distinct name words: 19148
10b: scanning label text (about a minute) ...


scan chunks:   0%|          | 0/97579 [00:00<?, ?it/s]

  words with precision < PRECISION_MIN: 3667
entity table: kept=39,939 | dropped as too generic=2,695
  sample of dropped (name, precision): [('DYSPHANIA AMBROSIOIDES', 0.0), ('SILTUXIMAB', 0.0), ('POMALIDOMIDE', 0.0), ('ETHANOL', 0.0), ('ETELCALCETIDE', 0.0), ('SOTORASIB', 0.0), ('PALBOCICLIB', 0.0), ('BINIMETINIB', 0.0), ('ANAKINRA', 0.0), ('DAROLUTAMIDE', 0.0), ('ECULIZUMAB', 0.0), ('DENATURED ETHANOL', 0.0)]
What should I avoid if I am pregnant and using lidocaine? 
   entities: [('LIDOCAINE', 'generic'), ('Lidocaine', 'brand'), ('Lidocaine', 'brand'), ('Lidocaine', 'brand'), ('Lidocaine', 'brand')] | intent: ['pregnancy_or_breast_feeding', 'warnings', 'do_not_use']
Is Aspercreme safe for children? 
   entities: [('Aspercreme', 'brand')] | intent: ['do_not_use', 'dosage_and_administration', 'warnings']
What can I take for headaches while pregnant? 
   entities: [] | intent: ['pregnancy_or_breast_feeding', 'warnings', 'indications_and_usage', 'purpose']


In [ ]:
# ===== STEP 11: retrieve = route -> rank drugs -> select chunks within token budget =====
def select_chunks(drug_id, wanted, variants, budget_chars):
    allowed = np.array([i for i in IDX_BY_DRUG.get(drug_id, []) if CHUNKS[i]["section"] in wanted])
    if len(allowed) == 0: return []
    ranked = hybrid_rank(variants, allowed, k=len(allowed))
    chosen, used, seen_sec = [], 0, set()
    for i, s, c in ranked:                                # pass 1: best chunk of every wanted section
        sec = CHUNKS[i]["section"]
        if sec not in seen_sec and used + len(CHUNKS[i]["text"]) <= budget_chars:
            chosen.append((i, c)); seen_sec.add(sec); used += len(CHUNKS[i]["text"])
    done = {i for i, _ in chosen}
    for i, s, c in ranked:                                # pass 2: fill remaining budget by relevance
        if i not in done and used + len(CHUNKS[i]["text"]) <= budget_chars:
            chosen.append((i, c)); used += len(CHUNKS[i]["text"])
    chosen.sort(key=lambda x: (SECTION_ORDER.index(CHUNKS[x[0]]["section"]) if CHUNKS[x[0]]["section"] in SECTION_ORDER else 99, x[0]))
    return chosen

def retrieve(query, verbose=True):
    variants = expand_queries(query)
    ents, intent = detect_entities(query), detect_intent(query)
    if ents:
        mode = "drug"; drug_order = [e["drug_id"] for e in ents]
        wanted = set(intent) if intent else set(DEFAULT_SECTIONS_DRUG)
    else:
        mode = "symptom"
        idxA = np.concatenate([IDX_BY_SECTION[s] for s in ("indications_and_usage", "purpose") if s in IDX_BY_SECTION])
        resA = hybrid_rank(variants, idxA, k=SYMPTOM_FETCH)      # symptom route: indications + purpose only
        resB = hybrid_rank(variants, None, k=SYMPTOM_FETCH)      # safety net: all sections (catches unknown brand names)
        ds = defaultdict(list)
        for i, s, c in resA: ds[CHUNKS[i]["drug_id"]].append(s)
        for i, s, c in resB: ds[CHUNKS[i]["drug_id"]].append(0.5 * s)
        agg = {d: sum(sorted(v, reverse=True)[:3]) for d, v in ds.items()}
        drug_order = [d for d, _ in sorted(agg.items(), key=lambda x: -x[1])][:MAX_DRUGS]
        wanted = set(intent) | {"indications_and_usage", "purpose"}
        if not intent: wanted |= {"warnings", "do_not_use"}
    if verbose:
        print(f"  mode={mode} | entities={[e['matched'] for e in ents]} | intent={intent} | wanted={sorted(wanted)}")
    per_drug = (TOTAL_CONTEXT_TOKENS * CHARS_PER_TOKEN) // max(1, len(drug_order))
    drugs, sources, top_cos = [], [], 0.0
    for d in drug_order:
        sel = select_chunks(d, wanted, variants, per_drug)
        have = {CHUNKS[i]["section"] for i in IDX_BY_DRUG.get(d, [])}
        missing = [s for s in intent if s not in have]
        drugs.append({"drug_id": d, "drug": GENERIC_NAME[d], "missing_sections": missing, "n_chunks": len(sel)})
        for i, c in sel:
            sources.append({"tag": f"S{len(sources)+1}", "idx": i, "drug": CHUNKS[i]["drug"], "section": CHUNKS[i]["section"],
                            "node": CHUNKS[i]["cid"], "cos": round(c, 3), "text": CHUNKS[i]["text"]})
            top_cos = max(top_cos, c)
        if verbose: print(f"  drug: {GENERIC_NAME[d]} | chunks={len(sel)} | missing={missing}")
    return {"mode": mode, "entities": [e["matched"] for e in ents], "intent": intent, "drugs": drugs,
            "sources": sources, "top_cos": round(top_cos, 3)}

In [ ]:
# ===== STEP 12: prompt, generation, abstention, support check =====
ABSTAIN_TEXT = "INSUFFICIENT FDA DATA: The retrieved label text does not contain this information."

def build_prompt(question, ret):
    parts = []
    for d in ret["drugs"]:
        parts.append(f"=== DRUG: {d['drug']} ===")
        if d["missing_sections"]:
            parts.append(f"NOTE: this label has NO text for: {', '.join(d['missing_sections'])}.")
        for s in [x for x in ret["sources"] if x["drug"] == d["drug"]]:
            parts.append(f"[{s['tag']}] ({s['section']}) {s['text']}")
    return ("You answer ONLY from the FDA drug label excerpts below.\nRULES:\n"
            "1. Use only the excerpts. Never use outside knowledge.\n"
            "2. Write short bullet points. Every bullet ends with its source tag(s), e.g. [S2] or [S1][S3].\n"
            "3. Keep drugs separate: start each bullet with the drug name.\n"
            f"4. If the excerpts do not answer the question, reply exactly: {ABSTAIN_TEXT}\n"
            "5. Give no medical advice beyond what the label says.\n\n"
            "EXCERPTS:\n" + "\n".join(parts) + f"\n\nQUESTION: {question}\n\nANSWER:\n")

def parse_claims(answer):
    claims = []
    for line in answer.splitlines():
        line = line.strip()
        if not line or ABSTAIN_TEXT[:20] in line: continue
        tags = re.findall(r"\[S(\d+)\]", line)
        text = re.sub(r"\[S\d+\]", "", line).lstrip("-*• ").strip()
        if tags or (re.match(r"^[-*•\d]", line) and len(text) > 25) or len(text) > 60:
            claims.append({"claim": text, "tags": [f"S{t}" for t in tags]})
    return claims

def support_check(claims, sources):
    by_tag = {s["tag"]: s for s in sources}
    for c in claims:
        if not c["tags"]: c["status"] = "no_citation"; continue
        bad = [t for t in c["tags"] if t not in by_tag]
        if bad: c["status"] = "invalid_citation"; continue
        evidence = "\n".join(by_tag[t]["text"] for t in c["tags"])
        v = llm(f"Label text:\n{evidence}\n\nClaim: {c['claim']}\n\nDoes the label text directly support the claim? "
                "Answer with one word: YES or NO.", temperature=0.0, num_predict=3)
        c["status"] = "supported" if v.upper().startswith("YES") else "unsupported"
    return claims

def answer_question(question, verbose=True):
    t0 = time.time()
    if verbose: print("\nQUESTION:", question)
    ret = retrieve(question, verbose)
    rec = {"question": question, "mode": ret["mode"], "entities": ret["entities"], "intent": ret["intent"],
           "drugs": [d["drug"] for d in ret["drugs"]], "sections_in_context": sorted({s["section"] for s in ret["sources"]}),
           "sources": [{k: v for k, v in s.items() if k != "text"} for s in ret["sources"]], "top_cos": ret["top_cos"],
           "context_chars": sum(len(s["text"]) for s in ret["sources"]), "claims": [], "unsupported_rate": None}
    reason = None
    if not ret["sources"]: reason = "no chunks retrieved"
    elif ret["mode"] == "symptom" and ret["top_cos"] < ABSTAIN_COS: reason = f"top cosine {ret['top_cos']} < {ABSTAIN_COS}"
    if reason:
        if verbose: print("  ABSTAIN before LLM:", reason)
        rec.update(raw_answer=ABSTAIN_TEXT, final_answer=ABSTAIN_TEXT, abstained=True, abstain_reason=reason)
    else:
        if verbose: print(f"  context={rec['context_chars']} chars, {len(ret['sources'])} chunks | generating ...")
        raw = llm(build_prompt(question, ret))
        rec["raw_answer"], rec["abstained"], rec["abstain_reason"] = raw, ABSTAIN_TEXT[:20] in raw, "llm" if ABSTAIN_TEXT[:20] in raw else None
        claims = support_check(parse_claims(raw), ret["sources"])
        rec["claims"] = claims
        if claims: rec["unsupported_rate"] = round(sum(c["status"] != "supported" for c in claims) / len(claims), 3)
        rec["final_answer"] = "\n".join(("- " if c["status"] == "supported" else f"- [FLAGGED: {c['status']}] ") + c["claim"] + " " + "".join(f"[{t}]" for t in c["tags"]) for c in claims) or raw
        if verbose:
            print("  RAW ANSWER:\n", raw)
            print("  SUPPORT CHECK:", Counter(c["status"] for c in claims), "| unsupported rate:", rec["unsupported_rate"])
    rec["seconds"] = round(time.time() - t0, 1)
    if verbose: print(f"  done in {rec['seconds']}s")
    return rec, ret

In [ ]:
# ===== STEP 13: quick demo on 3 questions (check everything works before the full QA run) =====
for q in ["What should I avoid if I am pregnant and using lidocaine?",
          "What can I take for headaches while pregnant?",
          "Is Aspercreme safe for children?"]:
    rec, ret = answer_question(q)
    print("-" * 80)


QUESTION: What should I avoid if I am pregnant and using lidocaine?
  mode=drug | entities=['LIDOCAINE', 'Lidocaine', 'Lidocaine', 'Lidocaine', 'Lidocaine'] | intent=['pregnancy_or_breast_feeding', 'warnings', 'do_not_use'] | wanted=['do_not_use', 'pregnancy_or_breast_feeding', 'warnings']
  drug: LIDOCAINE | chunks=3 | missing=[]
  drug: LIDOCAINE HYDROCHLORIDE | chunks=4 | missing=['pregnancy_or_breast_feeding', 'do_not_use']
  drug: LIDOCAINE 4% | chunks=3 | missing=['pregnancy_or_breast_feeding', 'do_not_use']
  drug: LIDOCAINE HYDROCHLORIDE AND EPINEPHRINE | chunks=4 | missing=['pregnancy_or_breast_feeding', 'do_not_use']
  drug: LIDOCAINE HYDROCHLORIDE AND EPINEPHRINE BITARTRATE | chunks=4 | missing=['pregnancy_or_breast_feeding', 'do_not_use']
  context=8396 chars, 18 chunks | generating ...
  RAW ANSWER:
 INSUFFICIENT FDA DATA: The retrieved label text does not contain information on what to avoid while using lidocaine while pregnant.
  SUPPORT CHECK: Counter() | unsupported r

In [ ]:
# ===== STEP 14: run the QA set (resumable, saved after every question) =====
import traceback

# --- paths: QA.json is in graph_rag/, so point to it explicitly ---
QA_PATH = "/content/drive/MyDrive/Research_kgTxagent/graph_rag/QA.json"
RESULTS_PATH = os.path.join(OUT_DIR, "qa_results_adv.json")

# --- guards: stop before running anything if paths/Drive are wrong ---
assert any(l.split()[1] == "/content/drive" for l in open("/proc/mounts")), "Drive is NOT mounted - re-run Step 3"
assert os.path.exists(QA_PATH), f"QA.json not found at {QA_PATH}"
print("QA file :", QA_PATH)
print("Results :", RESULTS_PATH)
print("Skipping prefixes:", SKIP_PREFIXES if SKIP_PREFIXES else "none (all test types run)")

QA = json.load(open(QA_PATH))
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
todo = [q for q in QA if not q.startswith(SKIP_PREFIXES) and q not in RESULTS]
print(f"QA questions: {len(QA)} | already done: {len(RESULTS)} | to run now: {len(todo)}")
print("To run:", todo)

def nm(s): return re.sub(r"[^A-Z0-9]+", " ", s.upper()).strip()
def gold_drugs(qa): return [m for t in qa["trace"] for m in re.findall(r"DRUG: (.*?) \| SOURCE", t)]

failed = []
for n, qid in enumerate(todo, start=1):
    qa = QA[qid]
    print(f"\n######## {qid}  ({n}/{len(todo)}) ########")
    try:
        rec, _ = answer_question(qa["question"], verbose=True)
    except Exception as e:
        print("  !! ERROR on", qid, ":", repr(e))
        traceback.print_exc()
        failed.append(qid); continue
    gold = [nm(g) for g in gold_drugs(qa)]; got = [nm(d) for d in rec["drugs"]]
    rec["gold_drugs"] = gold
    rec["drug_recall"] = round(sum(g in got for g in gold) / len(gold), 2) if gold else None
    rec["section_recall"] = round(len(set(qa["sections"]) & set(rec["sections_in_context"])) / len(qa["sections"]), 2) if qa["sections"] else None
    rec["answerable"], rec["answer_key"] = qa["answerable"], qa["answer"]
    rec["abstain_correct"] = (rec["abstained"] == (not qa["answerable"]))
    RESULTS[qid] = rec
    json.dump(RESULTS, open(RESULTS_PATH, "w"), indent=2)      # saved after every question
    print(f"  >> gold={gold} got={got} | drug_recall={rec['drug_recall']} section_recall={rec['section_recall']} abstain_correct={rec['abstain_correct']}")

print("\nQA run complete:", len(RESULTS), "results saved to", RESULTS_PATH)
if failed: print("Questions that errored (re-run this cell to retry them):", failed)

QA file : /content/drive/MyDrive/Research_kgTxagent/graph_rag/QA.json
Results : /content/drive/MyDrive/Research_kgTxagent/graph_rag/Adv_graphRAG/qa_results_adv.json
Skipping prefixes: none (all test types run)
QA questions: 60 | already done: 0 | to run now: 60
To run: ['T1-001', 'T1-002', 'T1-003', 'T1-004', 'T1-005', 'T1-006', 'T2-001', 'T2-002', 'T2-003', 'T2-004', 'T2-005', 'T2-006', 'T2-007', 'T2-008', 'T3-005', 'T3-006', 'T3-007', 'T3-008', 'T3-009', 'T4-001', 'T4-002', 'T4-003', 'T4-004', 'T4-005', 'T4-006', 'T4-007', 'T4-008', 'T4-009', 'T5-001', 'T5-002', 'T5-003', 'T5-004', 'T5-005', 'T5-006', 'T5-007', 'T5-008', 'T5-009', 'T6-001', 'T6-002', 'T6-003', 'T6-004', 'T6-005', 'T6-006', 'T6-007', 'T6-008', 'T6-009', 'T7-001', 'T7-002', 'T7-003', 'T7-004', 'T7-005', 'T8-001', 'T8-002', 'T8-003', 'T8-004', 'T8-005', 'T3-001', 'T3-002', 'T3-003', 'T3-004']

######## T1-001  (1/60) ########

QUESTION: What should I avoid if I am pregnant and using lidocaine?
  mode=drug | entities=['L

In [ ]:
# ===== STEP 15: summary by test type =====
df = pd.DataFrame([{"qid": k, "type": k.split("-")[0], **{x: v.get(x) for x in
      ["drug_recall", "section_recall", "abstained", "abstain_correct", "unsupported_rate", "top_cos", "seconds"]}}
      for k, v in RESULTS.items()])
print(df.groupby("type").agg(n=("qid", "count"), drug_recall=("drug_recall", "mean"), section_recall=("section_recall", "mean"),
                              abstain_correct=("abstain_correct", "mean"), unsupported_rate=("unsupported_rate", "mean"),
                              avg_seconds=("seconds", "mean")).round(2))
print("\nOverall drug_recall:", round(df.drug_recall.mean(), 2), "| section_recall:", round(df.section_recall.mean(), 2),
      "| abstain_correct:", round(df.abstain_correct.mean(), 2))
print("\nQuestions where the gold drug was NOT retrieved (check retrieval first):")
print(df[df.drug_recall < 1][["qid", "drug_recall"]].to_string(index=False))
print("\nNext: score each answer as correct / partially correct / wrong by comparing 'final_answer' with 'answer_key' in", RESULTS_PATH)

      n  drug_recall  section_recall  abstain_correct  unsupported_rate  \
type                                                                      
T1    6         0.67            0.62             0.67              0.15   
T2    8         0.00            0.67             0.38              0.44   
T3    9         0.67            0.76             0.56              0.14   
T4    9         0.00            1.00             0.67              0.40   
T5    9         0.56            0.83             0.44              0.43   
T6    9         1.00             NaN             0.89              0.50   
T7    5         1.00            1.00             0.80              0.30   
T8    5         0.50            0.83             0.60              0.43   

      avg_seconds  
type               
T1           5.85  
T2           7.22  
T3           4.68  
T4           9.47  
T5           7.57  
T6           1.22  
T7           6.30  
T8           4.54  

Overall drug_recall: 0.52 | section_recall: 0.81

In [ ]:
# ===== STEP 16: ask your own question =====
rec, ret = answer_question("Can I use Bactine Max on a child's scraped knee?")
print("\nFINAL ANSWER:\n", rec["final_answer"])
print("\nSOURCES:")
for s in rec["sources"]: print(" ", s["tag"], s["drug"], "|", s["section"], "|", s["node"], "| cos", s["cos"])


QUESTION: Can I use Bactine Max on a child's scraped knee?
  mode=drug | entities=['Bactine Max', 'Bactine Max', 'Bactine Max'] | intent=['do_not_use', 'dosage_and_administration', 'warnings'] | wanted=['do_not_use', 'dosage_and_administration', 'warnings']
  drug: LIDOCAINE | chunks=3 | missing=[]
  drug: BENZALKONIUM CHLORIDE AND LIDOCAINE HYDROCHLORIDE | chunks=2 | missing=['do_not_use']
  drug: BACTINE MAX WOUND WASH | chunks=3 | missing=[]
  context=1936 chars, 8 chunks | generating ...
  RAW ANSWER:
 Based on the FDA drug label excerpts, here is the answer:

* Bactine Max: [S6] Directions: Clean the affected area, apply a small amount of this product on the area 1 to 3 times daily; may be covered with a sterile bandage (let dry first).
* No specific warning or contraindication for using Bactine Max on a child's scraped knee.

So, the answer is: YES, you can use Bactine Max on a child's scraped knee, according to the label.
  SUPPORT CHECK: Counter({'no_citation': 2, 'supported':